In [ ]:

from fractions import Fraction

# --------------------------------------------------
# INDEPENDENCIA LINEAL DE VECTORES
# Método: Combinación lineal (sin determinantes)
# --------------------------------------------------

print("DETERMINACIÓN DE INDEPENDENCIA LINEAL")
print("Método: Definición de combinación lineal")
print("No se utilizarán determinantes.\n")

# 1. Solicitar la cantidad de vectores y componentes.
cantidad = int(input("¿Cuántos vectores deseas evaluar?: "))
componentes = int(input("¿Cuántas componentes tendrá cada vector?: "))

if cantidad < 1 or componentes < 1:
    raise ValueError("Ambas cantidades deben ser positivas.")

# 2. Solicitar los valores de cada vector.
vectores = []

for i in range(cantidad):
    while True:
        entrada = input(
            f"Ingresa el vector v{i+1} "
            f"({componentes} valores separados por espacios): "
        )

        try:
            valores = [Fraction(x) for x in entrada.split()]

            if len(valores) != componentes:
                print(f"Debes ingresar exactamente {componentes} valores.")
                continue

            vectores.append(valores)
            break

        except (ValueError, ZeroDivisionError):
            print("Ingresa números válidos. Ejemplo: 1 2 3")

# 3. Mostrar los vectores ingresados.
print("\nVECTORES INGRESADOS:")

for i, vector in enumerate(vectores):
    print(f"v{i+1} = {tuple(vector)}")

# 4. Construir el sistema:
# a1*v1 + a2*v2 + ... + an*vn = 0
# Cada vector se coloca como una columna.
matriz = [
    [vectores[j][i] for j in range(cantidad)]
    for i in range(componentes)
]

# 5. Resolver por eliminación de Gauss-Jordan.
# Este método NO utiliza determinantes.
fila_pivote = 0
columnas_pivote = []

for columna in range(cantidad):
    # Buscar una fila con elemento diferente de cero.
    pivote = next(
        (f for f in range(fila_pivote, componentes)
         if matriz[f][columna] != 0),
        None
    )

    if pivote is None:
        continue

    # Intercambiar filas.
    matriz[fila_pivote], matriz[pivote] = (
        matriz[pivote], matriz[fila_pivote]
    )

    # Convertir el pivote en 1.
    valor = matriz[fila_pivote][columna]
    matriz[fila_pivote] = [
        x / valor for x in matriz[fila_pivote]
    ]

    # Convertir los otros valores de la columna en 0.
    for f in range(componentes):
        if f != fila_pivote:
            factor = matriz[f][columna]

            matriz[f] = [
                matriz[f][j] - factor * matriz[fila_pivote][j]
                for j in range(cantidad)
            ]

    columnas_pivote.append(columna)
    fila_pivote += 1

    if fila_pivote == componentes:
        break

# 6. Analizar la solución del sistema homogéneo.
print("\nRESULTADO:")

if len(columnas_pivote) == cantidad:
    print("Los vectores son LINEALMENTE INDEPENDIENTES.")
    print("\n¿Por qué?")
    print("La única combinación lineal que produce")
    print("el vector cero tiene todos sus coeficientes")
    print("iguales a cero (solución trivial).")

else:
    print("Los vectores son LINEALMENTE DEPENDIENTES.")
    print("\n¿Por qué?")
    print("Existe una combinación lineal no trivial")
    print("que produce el vector cero.")

    # Encontrar una variable libre.
    libres = [
        j for j in range(cantidad)
        if j not in columnas_pivote
    ]

    # Asignar 1 a una variable libre.
    coeficientes = [Fraction(0) for _ in range(cantidad)]
    coeficientes[libres[0]] = Fraction(1)

    # Calcular los coeficientes de las variables pivote.
    for fila, columna in enumerate(columnas_pivote):
        coeficientes[columna] = -sum(
            matriz[fila][j] * coeficientes[j]
            for j in libres
        )

    print("\nCombinación lineal encontrada:")
    expresion = " + ".join(
        f"({coeficientes[i]})v{i+1}"
        for i in range(cantidad)
    )

    print(expresion, "= vector cero")
    print("\nAl menos un coeficiente es diferente de cero.")
    print("Por eso, el conjunto es linealmente dependiente.")


In [ ]:

%pip install sympy


In [2]:
import sympy as sp
from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    convert_xor,
    implicit_multiplication_application
)

# ORTOGONALIDAD DE MATRICES

print("DETERMINACIÓN DE ORTOGONALIDAD DE MATRICES")
print("\nRecordemos que una matriz es ortogonal")
print("cuando su transpuesta multiplicada por")
print("la matriz original es igual a la identidad.")
print("Condición: A^T * A = I\n")

print("Puedes utilizar números y expresiones como:")
print("cos(θ), -sin(θ), sqrt(2), x, 1/2")
print("Separa los elementos con comas.\n")

# Configurar el reconocimiento de expresiones matemáticas
transformaciones = standard_transformations + (
    convert_xor,
    implicit_multiplication_application
)

# Reconocer letras y funciones matemáticas
simbolos = {
    "θ": sp.Symbol("θ"),
    "α": sp.Symbol("α"),
    "β": sp.Symbol("β"),
    "theta": sp.Symbol("θ"),
    "sin": sp.sin,
    "cos": sp.cos,
    "tan": sp.tan,
    "sqrt": sp.sqrt
}

# Solicitar el tamaño de la matriz
n = int(input("¿De qué tamaño será tu matriz cuadrada?: "))

if n < 1:
    raise ValueError("El tamaño debe ser positivo.")

# Solicitar los elementos de la matriz
datos = []

for i in range(n):
    while True:
        entrada = input(f"Ingresa la fila {i+1}: ")

        try:
            partes = entrada.split(",")

            if len(partes) != n:
                print(f"Debes ingresar {n} elementos separados por comas.")
                continue

            fila = []

            for elemento in partes:
                valor = parse_expr(
                    elemento.strip(),
                    local_dict=simbolos,
                    transformations=transformaciones
                )
                fila.append(valor)

            datos.append(fila)
            break

        except Exception:
            print("Expresión no válida. Intenta nuevamente.")

# Crear la matriz
A = sp.Matrix(datos)

print("\nMATRIZ ORIGINAL:")
sp.pprint(A)

# Calcular la transpuesta
AT = A.T

print("\nMATRIZ TRANSPUESTA:")
sp.pprint(AT)

# Multiplicar la transpuesta por la original
resultado = AT * A

print("\nMULTIPLICACIÓN A^T * A:")
sp.pprint(resultado)

# Simplificar las operaciones
resultado = resultado.applyfunc(sp.trigsimp)
resultado = resultado.applyfunc(sp.simplify)

print("\nRESULTADO SIMPLIFICADO:")
sp.pprint(resultado)

# Crear la matriz identidad
identidad = sp.eye(n)

print("\nMATRIZ IDENTIDAD:")
sp.pprint(identidad)

# Comprobar la ortogonalidad
print("\nCONCLUSIÓN:")

if resultado.equals(identidad):
    print("La matriz es ORTOGONAL.")
    print("Porque A^T * A = I.")
else:
    diferencia = resultado - identidad

    if any(valor.equals(0) is None for valor in diferencia):
        print("No se pudo determinar simbólicamente")
        print("si la matriz es ortogonal.")
    else:
        print("La matriz NO es ortogonal.")
        print("Porque A^T * A no es igual a I.")


DETERMINACIÓN DE ORTOGONALIDAD DE MATRICES

Recordemos que una matriz es ortogonal
cuando su transpuesta multiplicada por
la matriz original es igual a la identidad.
Condición: A^T * A = I

Puedes utilizar números y expresiones como:
cos(θ), -sin(θ), sqrt(2), x, 1/2
Separa los elementos con comas.



¿De qué tamaño será tu matriz cuadrada?:  2
Ingresa la fila 1:  cos(x), -sin(x)
Ingresa la fila 2:  sin(x), cos(x)



MATRIZ ORIGINAL:
⎡cos(x)  -sin(x)⎤
⎢               ⎥
⎣sin(x)  cos(x) ⎦

MATRIZ TRANSPUESTA:
⎡cos(x)   sin(x)⎤
⎢               ⎥
⎣-sin(x)  cos(x)⎦

MULTIPLICACIÓN A^T * A:
⎡   2         2                      ⎤
⎢sin (x) + cos (x)          0        ⎥
⎢                                    ⎥
⎢                      2         2   ⎥
⎣        0          sin (x) + cos (x)⎦

RESULTADO SIMPLIFICADO:
⎡1  0⎤
⎢    ⎥
⎣0  1⎦

MATRIZ IDENTIDAD:
⎡1  0⎤
⎢    ⎥
⎣0  1⎦

CONCLUSIÓN:
La matriz es ORTOGONAL.
Porque A^T * A = I.


In [3]:

import sympy as sp

# --------------------------------------------
# TEMA 03: OPERACIONES DE MATRICES
# --------------------------------------------

print("OPERACIONES DE MATRICES")
print("\n¿Qué operación deseas realizar?")
print("1. Suma")
print("2. Resta")
print("3. Multiplicación")
print("4. Multiplicación por escalar")
print("5. Propiedades de las matrices")

# Pedimos al usuario seleccionar una operación
operacion = input("\nSelecciona una opción (1-5): ")


# FUNCIÓN PARA INGRESAR UNA MATRIZ
# Esta función evita repetir el mismo código.
def ingresar_matriz(nombre, filas, columnas):

    print("\nIngresa los valores de la matriz", nombre)
    print("Separa los valores con comas.")

    datos = []

    # Pedimos cada fila de la matriz
    for i in range(filas):

        while True:
            entrada = input(f"Fila {i+1}: ")

            try:
                # Convertimos los valores a expresiones
                fila = [
                    sp.sympify(x.strip(), rational=True)
                    for x in entrada.split(",")
                ]

                if len(fila) == columnas:
                    datos.append(fila)
                    break
                else:
                    print("Debes ingresar", columnas, "valores.")

            except Exception:
                print("Ingresa valores válidos.")

    # Convertimos los datos en una matriz
    matriz = sp.Matrix(datos)

    print("\nTu matriz", nombre, "es la siguiente:")
    sp.pprint(matriz)

    return matriz


# FUNCIÓN PARA MOSTRAR RESULTADOS
def mostrar_resultado(resultado):

    # Simplificamos expresiones matemáticas
    resultado = sp.simplify(sp.trigsimp(resultado))

    print("\nEl resultado es el siguiente:")
    sp.pprint(resultado)


# --------------------------------------------
# OPCIÓN 1: SUMA
# --------------------------------------------
if operacion == "1":

    print("\nSUMA DE MATRICES")

    filas = int(input("¿Cuántas filas tendrán?: "))
    columnas = int(input("¿Cuántas columnas tendrán?: "))

    A = ingresar_matriz("A", filas, columnas)
    B = ingresar_matriz("B", filas, columnas)

    print("\nOperación: A + B")

    resultado = A + B
    mostrar_resultado(resultado)


# --------------------------------------------
# OPCIÓN 2: RESTA
# --------------------------------------------
elif operacion == "2":

    print("\nRESTA DE MATRICES")

    filas = int(input("¿Cuántas filas tendrán?: "))
    columnas = int(input("¿Cuántas columnas tendrán?: "))

    A = ingresar_matriz("A", filas, columnas)
    B = ingresar_matriz("B", filas, columnas)

    print("\nOperación: A - B")

    resultado = A - B
    mostrar_resultado(resultado)


# --------------------------------------------
# OPCIÓN 3: MULTIPLICACIÓN
# --------------------------------------------
elif operacion == "3":

    print("\nMULTIPLICACIÓN DE MATRICES")

    filasA = int(input("Filas de A: "))
    columnasA = int(input("Columnas de A: "))

    filasB = int(input("Filas de B: "))
    columnasB = int(input("Columnas de B: "))

    # Para multiplicar, columnas de A = filas de B
    if columnasA != filasB:
        print("\nNo se pueden multiplicar estas matrices.")
        print("Las columnas de A deben ser iguales")
        print("a las filas de B.")

    else:
        A = ingresar_matriz("A", filasA, columnasA)
        B = ingresar_matriz("B", filasB, columnasB)

        print("\nOperación: A * B")

        resultado = A * B
        mostrar_resultado(resultado)


# --------------------------------------------
# OPCIÓN 4: MULTIPLICACIÓN POR ESCALAR
# --------------------------------------------
elif operacion == "4":

    print("\nMULTIPLICACIÓN POR ESCALAR")

    filas = int(input("Filas de A: "))
    columnas = int(input("Columnas de A: "))

    A = ingresar_matriz("A", filas, columnas)

    # El escalar es un solo valor
    escalar = sp.sympify(
        input("\nIngresa el escalar: "),
        rational=True
    )

    print("\nOperación:", escalar, "* A")

    resultado = escalar * A
    mostrar_resultado(resultado)


# --------------------------------------------
# OPCIÓN 5: PROPIEDADES
# --------------------------------------------
elif operacion == "5":

    print("\nPROPIEDADES DE LAS MATRICES")
    print("1. Distributiva")
    print("2. Asociativa")
    print("3. Conmutativa")

    propiedad = input("\n¿Qué propiedad deseas comprobar?: ")

    # Utilizaremos matrices cuadradas
    n = int(input("Tamaño de las matrices cuadradas: "))

    A = ingresar_matriz("A", n, n)
    B = ingresar_matriz("B", n, n)

    # PROPIEDAD DISTRIBUTIVA
    if propiedad == "1":

        C = ingresar_matriz("C", n, n)

        print("\nPROPIEDAD DISTRIBUTIVA")
        print("A * (B + C) = A*B + A*C")

        izquierda = A * (B + C)
        derecha = A*B + A*C

        print("\nLado izquierdo:")
        sp.pprint(izquierda)

        print("\nLado derecho:")
        sp.pprint(derecha)

        if izquierda.equals(derecha):
            print("\nSe cumple la propiedad distributiva.")
        else:
            print("\nNo se cumple la propiedad.")


    # PROPIEDAD ASOCIATIVA
    elif propiedad == "2":

        C = ingresar_matriz("C", n, n)

        print("\nPROPIEDAD ASOCIATIVA")
        print("(A*B)*C = A*(B*C)")

        izquierda = (A * B) * C
        derecha = A * (B * C)

        print("\nLado izquierdo:")
        sp.pprint(izquierda)

        print("\nLado derecho:")
        sp.pprint(derecha)

        if izquierda.equals(derecha):
            print("\nSe cumple la propiedad asociativa.")
        else:
            print("\nNo se cumple la propiedad.")


    # PROPIEDAD CONMUTATIVA
    elif propiedad == "3":

        print("\nPROPIEDAD CONMUTATIVA")

        # Comprobar conmutatividad de la suma
        print("\nComprobación de suma: A+B = B+A")

        izquierda = A + B
        derecha = B + A

        print("\nA + B:")
        sp.pprint(izquierda)

        print("\nB + A:")
        sp.pprint(derecha)

        if izquierda.equals(derecha):
            print("La suma es conmutativa.")

        # Comprobar conmutatividad de multiplicación
        print("\nComprobación: A*B = B*A")

        izquierda = A * B
        derecha = B * A

        print("\nA * B:")
        sp.pprint(izquierda)

        print("\nB * A:")
        sp.pprint(derecha)

        if izquierda.equals(derecha):
            print("Estas matrices sí conmutan al multiplicarse.")
        else:
            print("Estas matrices NO conmutan al multiplicarse.")
            print("La multiplicación de matrices")
            print("no es conmutativa en general.")

    else:
        print("Propiedad no válida.")


# OPCIÓN INCORRECTA
else:
    print("\nOpción no válida. Ingresa un número del 1 al 5.")


OPERACIONES DE MATRICES

¿Qué operación deseas realizar?
1. Suma
2. Resta
3. Multiplicación
4. Multiplicación por escalar
5. Propiedades de las matrices



Selecciona una opción (1-5):  1



SUMA DE MATRICES


¿Cuántas filas tendrán?:  3
¿Cuántas columnas tendrán?:  4



Ingresa los valores de la matriz A
Separa los valores con comas.


...


Tu matriz A es la siguiente:
⎡2  4  5  6⎤
⎢          ⎥
⎢4  5  6  4⎥
⎢          ⎥
⎣5  3  2  2⎦

Ingresa los valores de la matriz B
Separa los valores con comas.


Fila 1:  3,4,6,7,6


Debes ingresar 4 valores.


Fila 1:  3,4,3,2
Fila 2:  3,5,2,4
Fila 3:  3,5,2,4



Tu matriz B es la siguiente:
⎡3  4  3  2⎤
⎢          ⎥
⎢3  5  2  4⎥
⎢          ⎥
⎣3  5  2  4⎦

Operación: A + B

El resultado es el siguiente:
⎡5  8   8  8⎤
⎢           ⎥
⎢7  10  8  8⎥
⎢           ⎥
⎣8  8   4  6⎦


In [2]:

import sympy as sp

# --------------------------------------------
# TEMA 04: COMBINACIÓN LINEAL DE VECTORES
# --------------------------------------------

print("COMBINACIÓN LINEAL DE VECTORES")

print("\nRecordemos que una combinación lineal")
print("consiste en multiplicar uno o varios vectores")
print("por números llamados escalares y sumar")
print("los resultados obtenidos.")

print("\nForma general de una combinación lineal:")
print("a1*v1 + a2*v2 + ... + an*vn")

print("\nDonde:")
print("v1, v2, ..., vn son los vectores.")
print("a1, a2, ..., an son los escalares.")

print("\nEjemplo: 4v1 - 4v2")
print("Puedes utilizar números, fracciones y símbolos.")
print("Separa las componentes con comas.\n")


# 1. Solicitar cantidad de vectores
cantidad = int(input("¿Cuántos vectores deseas combinar?: "))

# 2. Solicitar cantidad de componentes
componentes = int(input("¿Cuántas componentes tendrá cada vector?: "))

if cantidad < 1 or componentes < 1:
    raise ValueError("Las cantidades deben ser positivas.")


# 3. Crear listas para guardar los datos
vectores = []
escalares = []


# 4. Solicitar los vectores
for i in range(cantidad):

    print("\nVECTOR", i + 1)

    while True:
        entrada = input(
            f"Ingresa las {componentes} componentes "
            f"del vector v{i+1}: "
        )

        try:
            valores = []

            for numero in entrada.split(","):
                valor = sp.sympify(numero.strip(), rational=True)
                valores.append(valor)

            if len(valores) != componentes:
                print("Debes ingresar", componentes, "valores.")
                continue

            # Convertir los valores a un vector columna
            vector = sp.Matrix(valores)

            vectores.append(vector)
            break

        except Exception:
            print("Valores incorrectos. Intenta nuevamente.")


    # 5. Solicitar el escalar de cada vector
    while True:
        try:
            escalar = sp.sympify(
                input(f"Ingresa el escalar de v{i+1}: "),
                rational=True
            )

            escalar = sp.simplify(escalar)

            if escalar.is_number is False and escalar.is_commutative is False:
                print("El escalar no es válido.")
                continue

            escalares.append(escalar)
            break

        except Exception:
            print("Escalar incorrecto. Intenta nuevamente.")


# 6. Mostrar los vectores ingresados
print("\nVECTORES INGRESADOS:")

for i in range(cantidad):

    print(f"\nTu vector v{i+1} es el siguiente:")
    sp.pprint(vectores[i])

    print("Su escalar es:")
    sp.pprint(escalares[i])


# 7. Mostrar la combinación lineal
print("\nCOMBINACIÓN LINEAL A RESOLVER:")

operacion = ""

for i in range(cantidad):

    if i > 0:
        operacion += " + "

    operacion += f"({escalares[i]})v{i+1}"

print(operacion)


# 8. Crear el vector resultado con ceros
resultado = sp.zeros(componentes, 1)


# 9. Realizar el procedimiento
print("\nPROCEDIMIENTO:")

for i in range(cantidad):

    print(f"\nPaso {i+1}: Multiplicar v{i+1} "
          f"por el escalar {escalares[i]}")

    print("\nVector original:")
    sp.pprint(vectores[i])

    print("\nEscalar:")
    sp.pprint(escalares[i])

    # Multiplicar el vector por su escalar
    multiplicado = escalares[i] * vectores[i]

    print("\nResultado de la multiplicación:")
    sp.pprint(multiplicado)

    # Sumar al resultado acumulado
    resultado = resultado + multiplicado

    print("\nResultado acumulado:")
    sp.pprint(resultado)


# 10. Simplificar el resultado final
resultado = resultado.applyfunc(sp.trigsimp)
resultado = resultado.applyfunc(sp.simplify)


# 11. Mostrar el resultado
print("\n----------------------------------")
print("EL RESULTADO ES EL SIGUIENTE:")
sp.pprint(resultado)
print("----------------------------------")


# 12. Mostrar conclusión
print("\nCONCLUSIÓN:")
print("La combinación lineal se obtuvo")
print("multiplicando cada vector por su escalar")
print("y sumando las componentes correspondientes.")


COMBINACIÓN LINEAL DE VECTORES

Recordemos que una combinación lineal
consiste en multiplicar uno o varios vectores
por números llamados escalares y sumar
los resultados obtenidos.

Forma general de una combinación lineal:
a1*v1 + a2*v2 + ... + an*vn

Donde:
v1, v2, ..., vn son los vectores.
a1, a2, ..., an son los escalares.

Ejemplo: 4v1 - 4v2
Puedes utilizar números, fracciones y símbolos.
Separa las componentes con comas.



¿Cuántos vectores deseas combinar?:  2
¿Cuántas componentes tendrá cada vector?:  2



VECTOR 1


Ingresa las 2 componentes del vector v1:  3, 1
Ingresa el escalar de v1:  4



VECTOR 2


Ingresa las 2 componentes del vector v2:  2, 5
Ingresa el escalar de v2:  -2



VECTORES INGRESADOS:

Tu vector v1 es el siguiente:
⎡3⎤
⎢ ⎥
⎣1⎦
Su escalar es:
4

Tu vector v2 es el siguiente:
⎡2⎤
⎢ ⎥
⎣5⎦
Su escalar es:
-2

COMBINACIÓN LINEAL A RESOLVER:
(4)v1 + (-2)v2

PROCEDIMIENTO:

Paso 1: Multiplicar v1 por el escalar 4

Vector original:
⎡3⎤
⎢ ⎥
⎣1⎦

Escalar:
4

Resultado de la multiplicación:
⎡12⎤
⎢  ⎥
⎣4 ⎦

Resultado acumulado:
⎡12⎤
⎢  ⎥
⎣4 ⎦

Paso 2: Multiplicar v2 por el escalar -2

Vector original:
⎡2⎤
⎢ ⎥
⎣5⎦

Escalar:
-2

Resultado de la multiplicación:
⎡-4 ⎤
⎢   ⎥
⎣-10⎦

Resultado acumulado:
⎡8 ⎤
⎢  ⎥
⎣-6⎦

----------------------------------
EL RESULTADO ES EL SIGUIENTE:
⎡8 ⎤
⎢  ⎥
⎣-6⎦
----------------------------------

CONCLUSIÓN:
La combinación lineal se obtuvo
multiplicando cada vector por su escalar
y sumando las componentes correspondientes.


In [3]:

import sympy as sp

# --------------------------------------------------
# TEMA 05: SOLUCIÓN PARTICULAR Y ESPACIO NULO
# --------------------------------------------------

print("SOLUCIÓN GENERAL DE SISTEMAS LINEALES")

print("\nRecordemos que un sistema de ecuaciones")
print("lineales se representa mediante:")
print("A * x = b")

print("\nCuando el sistema tiene soluciones,")
print("su solución general se expresa como:")
print("x = xp + xh")

print("\nDonde:")
print("xp = Solución particular de A*x = b")
print("xh = Solución del sistema A*x = 0")

print("\nSi existen variables libres:")
print("x = xp + λ1*v1 + λ2*v2 + ...")

print("\nPuedes utilizar números, fracciones")
print("y expresiones matemáticas.")
print("Separa los elementos con comas.\n")


# 1. SOLICITAR LAS DIMENSIONES DE A

filas = int(input("¿Cuántas ecuaciones tiene el sistema?: "))
columnas = int(input("¿Cuántas incógnitas tiene?: "))

if filas < 1 or columnas < 1:
    raise ValueError("Las cantidades deben ser positivas.")


# 2. SOLICITAR LA MATRIZ A

print("\nINGRESAR MATRIZ A")
print("Ingresa los coeficientes de cada ecuación.")

datos = []

for i in range(filas):

    while True:
        entrada = input(f"Ingresa la fila {i+1}: ")

        try:
            valores = [
                sp.sympify(x.strip(), rational=True)
                for x in entrada.split(",")
            ]

            if len(valores) != columnas:
                print("Debes ingresar", columnas, "valores.")
                continue

            datos.append(valores)
            break

        except Exception:
            print("Expresión incorrecta. Intenta nuevamente.")


# Crear la matriz A
A = sp.Matrix(datos)

print("\nTU MATRIZ A ES LA SIGUIENTE:")
sp.pprint(A)


# 3. SOLICITAR EL VECTOR b

print("\nINGRESAR VECTOR b")
print("Ingresa los resultados de las ecuaciones.")

while True:

    entrada = input(f"Ingresa {filas} valores separados por comas: ")

    try:
        valores = [
            sp.sympify(x.strip(), rational=True)
            for x in entrada.split(",")
        ]

        if len(valores) != filas:
            print("Debes ingresar", filas, "valores.")
            continue

        b = sp.Matrix(valores)
        break

    except Exception:
        print("Expresión incorrecta. Intenta nuevamente.")


print("\nTU VECTOR b ES EL SIGUIENTE:")
sp.pprint(b)


# 4. MOSTRAR EL SISTEMA

print("\nSISTEMA DE ECUACIONES:")

variables = sp.symbols(f"x1:{columnas+1}")
x = sp.Matrix(variables)

ecuaciones = A * x

for i in range(filas):
    print(sp.Eq(ecuaciones[i], b[i], evaluate=False))


# 5. VERIFICAR SI EXISTEN SOLUCIONES

print("\nANÁLISIS DEL SISTEMA:")

rango_A = A.rank()
rango_ampliada = A.row_join(b).rank()

if rango_A != rango_ampliada:

    print("\nEL SISTEMA NO TIENE SOLUCIÓN.")
    print("No existe un vector x que satisfaga A*x = b.")
    print("Por lo tanto, no hay solución general.")

else:

    # 6. ENCONTRAR UNA SOLUCIÓN PARTICULAR

    print("\nPASO 1: ENCONTRAR UNA SOLUCIÓN PARTICULAR")

    # Resolver el sistema A*x = b
    solucion, parametros = A.gauss_jordan_solve(b)

    # Asignar cero a los parámetros libres
    sustituciones = {}

    for parametro in parametros:
        sustituciones[parametro] = 0

    xp = solucion.subs(sustituciones)

    print("\nUna solución particular xp es:")
    sp.pprint(xp)

    print("\nComprobación A*xp = b:")
    sp.pprint(A * xp)


    # 7. ENCONTRAR EL ESPACIO NULO

    print("\nPASO 2: ENCONTRAR EL ESPACIO NULO")
    print("Resolvemos el sistema homogéneo A*x = 0")

    espacio_nulo = A.nullspace()

    if len(espacio_nulo) == 0:

        print("\nEl espacio nulo contiene")
        print("únicamente al vector cero.")

        print("\nSOLUCIÓN ÚNICA:")
        sp.pprint(xp)

        print("\nCONCLUSIÓN:")
        print("El sistema tiene una única solución.")
        print("No existen parámetros libres.")

    else:

        print("\nBASE DEL ESPACIO NULO:")

        for i in range(len(espacio_nulo)):

            print(f"\nVector v{i+1}:")
            sp.pprint(espacio_nulo[i])

            print("Comprobación A*v = 0:")
            sp.pprint(A * espacio_nulo[i])


        # 8. CONSTRUIR LA SOLUCIÓN GENERAL

        print("\nPASO 3: CONSTRUIR LA SOLUCIÓN GENERAL")

        print("\nForma general:")
        print("x = xp + λ1*v1 + λ2*v2 + ...")

        print("\nSolución particular xp:")
        sp.pprint(xp)

        # Crear los parámetros libres
        lambdas = sp.symbols(
            f"lambda1:{len(espacio_nulo)+1}"
        )

        # Comenzar con la solución particular
        solucion_general = xp.copy()

        for i in range(len(espacio_nulo)):

            print(f"\nParámetro lambda{i+1}:")
            sp.pprint(lambdas[i])

            print(f"Vector v{i+1}:")
            sp.pprint(espacio_nulo[i])

            solucion_general += (
                lambdas[i] * espacio_nulo[i]
            )


        # 9. MOSTRAR RESULTADO FINAL

        print("\n----------------------------------")
        print("SOLUCIÓN GENERAL DEL SISTEMA:")
        sp.pprint(solucion_general)
        print("----------------------------------")


        # 10. COMPROBAR LA SOLUCIÓN GENERAL

        print("\nCOMPROBACIÓN:")
        print("Multiplicamos A por la solución general.")

        comprobacion = A * solucion_general

        comprobacion = comprobacion.applyfunc(sp.simplify)

        print("\nResultado de A*x:")
        sp.pprint(comprobacion)

        print("\nVector b original:")
        sp.pprint(b)


        # 11. MOSTRAR CONCLUSIÓN

        print("\nCONCLUSIÓN:")

        if comprobacion.equals(b):

            print("El sistema tiene infinitas soluciones.")
            print("La solución general es la suma de")
            print("una solución particular y una")
            print("combinación lineal de los vectores")
            print("que forman la base del espacio nulo.")

            print("\nSe cumple la condición:")
            print("A*x = b")

        else:
            print("No se pudo verificar la solución.")


SOLUCIÓN GENERAL DE SISTEMAS LINEALES

Recordemos que un sistema de ecuaciones
lineales se representa mediante:
A * x = b

Cuando el sistema tiene soluciones,
su solución general se expresa como:
x = xp + xh

Donde:
xp = Solución particular de A*x = b
xh = Solución del sistema A*x = 0

Si existen variables libres:
x = xp + λ1*v1 + λ2*v2 + ...

Puedes utilizar números, fracciones
y expresiones matemáticas.
Separa los elementos con comas.



¿Cuántas ecuaciones tiene el sistema?:  2
¿Cuántas incógnitas tiene?:  4



INGRESAR MATRIZ A
Ingresa los coeficientes de cada ecuación.


Ingresa la fila 1:  1, 0, 8, -4
Ingresa la fila 2:  0, 1, 2, 12



TU MATRIZ A ES LA SIGUIENTE:
⎡1  0  8  -4⎤
⎢           ⎥
⎣0  1  2  12⎦

INGRESAR VECTOR b
Ingresa los resultados de las ecuaciones.


Ingresa 2 valores separados por comas:  42, 8



TU VECTOR b ES EL SIGUIENTE:
⎡42⎤
⎢  ⎥
⎣8 ⎦

SISTEMA DE ECUACIONES:
Eq(x1 + 8*x3 - 4*x4, 42)
Eq(x2 + 2*x3 + 12*x4, 8)

ANÁLISIS DEL SISTEMA:

PASO 1: ENCONTRAR UNA SOLUCIÓN PARTICULAR

Una solución particular xp es:
⎡42⎤
⎢  ⎥
⎢8 ⎥
⎢  ⎥
⎢0 ⎥
⎢  ⎥
⎣0 ⎦

Comprobación A*xp = b:
⎡42⎤
⎢  ⎥
⎣8 ⎦

PASO 2: ENCONTRAR EL ESPACIO NULO
Resolvemos el sistema homogéneo A*x = 0

BASE DEL ESPACIO NULO:

Vector v1:
⎡-8⎤
⎢  ⎥
⎢-2⎥
⎢  ⎥
⎢1 ⎥
⎢  ⎥
⎣0 ⎦
Comprobación A*v = 0:
⎡0⎤
⎢ ⎥
⎣0⎦

Vector v2:
⎡ 4 ⎤
⎢   ⎥
⎢-12⎥
⎢   ⎥
⎢ 0 ⎥
⎢   ⎥
⎣ 1 ⎦
Comprobación A*v = 0:
⎡0⎤
⎢ ⎥
⎣0⎦

PASO 3: CONSTRUIR LA SOLUCIÓN GENERAL

Forma general:
x = xp + λ1*v1 + λ2*v2 + ...

Solución particular xp:
⎡42⎤
⎢  ⎥
⎢8 ⎥
⎢  ⎥
⎢0 ⎥
⎢  ⎥
⎣0 ⎦

Parámetro lambda1:
λ₁
Vector v1:
⎡-8⎤
⎢  ⎥
⎢-2⎥
⎢  ⎥
⎢1 ⎥
⎢  ⎥
⎣0 ⎦

Parámetro lambda2:
λ₂
Vector v2:
⎡ 4 ⎤
⎢   ⎥
⎢-12⎥
⎢   ⎥
⎢ 0 ⎥
⎢   ⎥
⎣ 1 ⎦

----------------------------------
SOLUCIÓN GENERAL DEL SISTEMA:
⎡-8⋅λ₁ + 4⋅λ₂ + 42⎤
⎢                 ⎥
⎢-2⋅λ₁ - 12⋅λ₂ + 8⎥
⎢                 ⎥
⎢     

In [4]:

import sympy as sp

# --------------------------------------------------
# TEMA 06: INVERSA DE UNA MATRIZ
# --------------------------------------------------

print("DETERMINACIÓN DE LA INVERSA DE UNA MATRIZ")

print("\nRecordemos que una matriz tiene inversa")
print("cuando al multiplicarla por su inversa")
print("obtenemos la matriz identidad.")

print("\nCondición:")
print("A * A^(-1) = I")

print("\nPara que una matriz tenga inversa:")
print("1. Debe ser cuadrada.")
print("2. Su determinante debe ser diferente de cero.")

print("\nSi det(A) = 0, la matriz es singular.")
print("Una matriz singular NO tiene inversa.")

print("\nPuedes utilizar números y expresiones como:")
print("3, -2, 1/2, sqrt(2), x, cos(θ)")
print("Separa los elementos con comas.\n")


# 1. SOLICITAR EL TAMAÑO DE LA MATRIZ

n = int(input("¿De qué tamaño será tu matriz cuadrada?: "))

if n < 1:
    raise ValueError("El tamaño debe ser positivo.")


# 2. SOLICITAR LOS VALORES DE LA MATRIZ

datos = []

print("\nINGRESAR MATRIZ A")

for i in range(n):

    while True:

        entrada = input(f"Ingresa la fila {i+1}: ")

        try:
            valores = []

            for elemento in entrada.split(","):
                valor = sp.sympify(
                    elemento.strip(),
                    rational=True
                )
                valores.append(valor)

            if len(valores) != n:
                print("Debes ingresar", n, "valores.")
                continue

            datos.append(valores)
            break

        except Exception:
            print("Expresión incorrecta. Intenta nuevamente.")


# 3. CREAR Y MOSTRAR LA MATRIZ ORIGINAL

A = sp.Matrix(datos)

print("\nTU MATRIZ A ES LA SIGUIENTE:")
sp.pprint(A)


# 4. CREAR LA MATRIZ IDENTIDAD

I = sp.eye(n)

print("\nMATRIZ IDENTIDAD:")
sp.pprint(I)


# 5. CALCULAR EL DETERMINANTE

print("\nPASO 1: CALCULAR EL DETERMINANTE")

determinante = sp.simplify(A.det())

print("\nDeterminante de A:")
sp.pprint(determinante)


# 6. COMPROBAR SI TIENE INVERSA

print("\nANÁLISIS DE LA MATRIZ:")

if determinante.equals(0) is True:

    print("\nLA MATRIZ NO TIENE INVERSA.")

    print("\n¿Por qué?")
    print("Porque su determinante es igual a cero.")

    print("\nCONCLUSIÓN:")
    print("La matriz es SINGULAR.")
    print("Por lo tanto, no existe A^(-1).")

else:

    # Si hay variables, la inversa puede depender
    # de los valores que tomen esas variables.
    if determinante.free_symbols:
        print("\nLa inversa requiere que det(A) != 0.")
        print("El resultado puede depender de variables.")

    print("\nPASO 2: MÉTODO DE GAUSS-JORDAN")

    # Construir la matriz aumentada [A | I]
    aumentada = A.row_join(I)

    print("\nMatriz aumentada [A | I]:")
    sp.pprint(aumentada)

    # Aplicar reducción por filas
    reducida, pivotes = aumentada.rref()

    print("\nMatriz reducida:")
    sp.pprint(reducida)

    # Extraer la parte derecha de la matriz
    inversa = reducida[:, n:]

    # Simplificar las expresiones
    inversa = inversa.applyfunc(sp.simplify)

    print("\nPASO 3: OBTENER LA INVERSA")

    print("\nLA MATRIZ INVERSA A^(-1) ES:")
    sp.pprint(inversa)


    # 7. COMPROBAR LA INVERSA

    print("\nPASO 4: COMPROBACIÓN")

    print("\nMultiplicamos A * A^(-1):")

    comprobacion = A * inversa

    comprobacion = comprobacion.applyfunc(sp.simplify)

    sp.pprint(comprobacion)

    print("\nMatriz identidad:")
    sp.pprint(I)


    # 8. MOSTRAR CONCLUSIÓN

    print("\nCONCLUSIÓN:")

    diferencia = comprobacion - I

    if all(
        elemento.equals(0) is True
        for elemento in diferencia
    ):

        print("La matriz tiene inversa.")
        print("Se cumple la condición:")
        print("A * A^(-1) = I")

        if determinante.free_symbols:
            print("Siempre que det(A) sea diferente de cero.")

    else:
        print("No se pudo verificar simbólicamente")
        print("la igualdad con la matriz identidad.")



DETERMINACIÓN DE LA INVERSA DE UNA MATRIZ

Recordemos que una matriz tiene inversa
cuando al multiplicarla por su inversa
obtenemos la matriz identidad.

Condición:
A * A^(-1) = I

Para que una matriz tenga inversa:
1. Debe ser cuadrada.
2. Su determinante debe ser diferente de cero.

Si det(A) = 0, la matriz es singular.
Una matriz singular NO tiene inversa.

Puedes utilizar números y expresiones como:
3, -2, 1/2, sqrt(2), x, cos(θ)
Separa los elementos con comas.



¿De qué tamaño será tu matriz cuadrada?:  4



INGRESAR MATRIZ A


Ingresa la fila 1:  1, 0, 2, 0
Ingresa la fila 2:  1, 1, 0, 0
Ingresa la fila 3:  1, 0, 0, 1
Ingresa la fila 4:  1, 1, 1, 1



TU MATRIZ A ES LA SIGUIENTE:
⎡1  0  2  0⎤
⎢          ⎥
⎢1  1  0  0⎥
⎢          ⎥
⎢1  0  0  1⎥
⎢          ⎥
⎣1  1  1  1⎦

MATRIZ IDENTIDAD:
⎡1  0  0  0⎤
⎢          ⎥
⎢0  1  0  0⎥
⎢          ⎥
⎢0  0  1  0⎥
⎢          ⎥
⎣0  0  0  1⎦

PASO 1: CALCULAR EL DETERMINANTE

Determinante de A:
-3

ANÁLISIS DE LA MATRIZ:

PASO 2: MÉTODO DE GAUSS-JORDAN

Matriz aumentada [A | I]:
⎡1  0  2  0  1  0  0  0⎤
⎢                      ⎥
⎢1  1  0  0  0  1  0  0⎥
⎢                      ⎥
⎢1  0  0  1  0  0  1  0⎥
⎢                      ⎥
⎣1  1  1  1  0  0  0  1⎦

Matriz reducida:
⎡1  0  0  0  1/3   2/3   2/3   -2/3⎤
⎢                                  ⎥
⎢0  1  0  0  -1/3  1/3   -2/3  2/3 ⎥
⎢                                  ⎥
⎢0  0  1  0  1/3   -1/3  -1/3  1/3 ⎥
⎢                                  ⎥
⎣0  0  0  1  -1/3  -2/3  1/3   2/3 ⎦

PASO 3: OBTENER LA INVERSA

LA MATRIZ INVERSA A^(-1) ES:
⎡1/3   2/3   2/3   -2/3⎤
⎢                      ⎥
⎢-1/3  1/3   -2/3  2/3 ⎥
⎢                      ⎥
⎢1/3   -1/3  -1/3  